# Chapter 17 Lab: Reliability: Bias, Robustness, Privacy, and Safety (`ch11`)

This notebook explores the pillars of reliable machine learning:
- Subgroup evaluation and performance disparity
- Fairness metrics: demographic parity vs. equalized odds
- Robustness under distribution shift
- Spurious correlations and shortcuts
- Calibration degradation under shift

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Subgroup Evaluation

Generate data with a protected attribute (group membership: A or B). Train a classifier and compute accuracy, precision, and recall separately for each group to reveal disparities.

In [ ]:
np.random.seed(42)

# Create two groups with different class distributions
# Group A: balanced
X_A, y_A = make_classification(
    n_samples=300,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    weights=[0.5, 0.5],
    random_state=42
)

# Group B: imbalanced (more positive class)
X_B, y_B = make_classification(
    n_samples=300,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    weights=[0.3, 0.7],  # More positive class
    random_state=43
)

# Combine and create group labels
X_combined = np.vstack([X_A, X_B])
y_combined = np.hstack([y_A, y_B])
group_labels = np.hstack([np.zeros(300), np.ones(300)])  # 0=Group A, 1=Group B

# Train/test split (stratify by group to keep balance)
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_combined, y_combined, group_labels, test_size=0.3, random_state=42
)

# Train model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

# Overall metrics
overall_acc = accuracy_score(y_test, y_pred)
overall_prec = precision_score(y_test, y_pred, zero_division=0)
overall_rec = recall_score(y_test, y_pred, zero_division=0)

# Group-specific metrics
mask_A = groups_test == 0
mask_B = groups_test == 1

acc_A = accuracy_score(y_test[mask_A], y_pred[mask_A])
prec_A = precision_score(y_test[mask_A], y_pred[mask_A], zero_division=0)
rec_A = recall_score(y_test[mask_A], y_pred[mask_A], zero_division=0)

acc_B = accuracy_score(y_test[mask_B], y_pred[mask_B])
prec_B = precision_score(y_test[mask_B], y_pred[mask_B], zero_division=0)
rec_B = recall_score(y_test[mask_B], y_pred[mask_B], zero_division=0)

print("Subgroup Evaluation Results")
print("=" * 60)
print(f"\nOverall Performance:")
print(f"  Accuracy:  {overall_acc:.4f}")
print(f"  Precision: {overall_prec:.4f}")
print(f"  Recall:    {overall_rec:.4f}")

print(f"\nGroup A Performance (n={mask_A.sum()}):")
print(f"  Accuracy:  {acc_A:.4f}")
print(f"  Precision: {prec_A:.4f}")
print(f"  Recall:    {rec_A:.4f}")

print(f"\nGroup B Performance (n={mask_B.sum()}):")
print(f"  Accuracy:  {acc_B:.4f}")
print(f"  Precision: {prec_B:.4f}")
print(f"  Recall:    {rec_B:.4f}")

print(f"\nDisparity (Group B - Group A):")
print(f"  Accuracy:  {(acc_B - acc_A):+.4f}")
print(f"  Precision: {(prec_B - prec_A):+.4f}")
print(f"  Recall:    {(rec_B - rec_A):+.4f}")

In [ ]:
# Visualize subgroup performance
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Grouped bar chart
ax = axes[0]
metrics = ['Accuracy', 'Precision', 'Recall']
group_A_vals = [acc_A, prec_A, rec_A]
group_B_vals = [acc_B, prec_B, rec_B]
overall_vals = [overall_acc, overall_prec, overall_rec]

x = np.arange(len(metrics))
width = 0.25

bars1 = ax.bar(x - width, group_A_vals, width, label='Group A', color='#2ca02c', alpha=0.7)
bars2 = ax.bar(x, group_B_vals, width, label='Group B', color='#d62728', alpha=0.7)
bars3 = ax.bar(x + width, overall_vals, width, label='Overall', color='#1f77b4', alpha=0.7)

ax.set_ylabel('Score', fontsize=11)
ax.set_title('Subgroup Performance: A vs. B vs. Overall', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=10)
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# Disparity heatmap
ax = axes[1]
disparity_matrix = np.array([
    [acc_A - overall_acc, acc_B - overall_acc],
    [prec_A - overall_prec, prec_B - overall_prec],
    [rec_A - overall_rec, rec_B - overall_rec]
])

im = ax.imshow(disparity_matrix, cmap='RdBu_r', aspect='auto', vmin=-0.15, vmax=0.15)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(['Group A', 'Group B'])
ax.set_yticklabels(['Accuracy', 'Precision', 'Recall'])
ax.set_title('Disparity from Overall Mean', fontweight='bold')

for i in range(3):
    for j in range(2):
        text = ax.text(j, i, f'{disparity_matrix[i, j]:+.3f}',
                       ha='center', va='center', color='white' if abs(disparity_matrix[i, j]) > 0.07 else 'black',
                       fontsize=11, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Difference from Overall', fontsize=10)

plt.tight_layout()
plt.show()

## 2. Demographic Parity vs. Equalized Odds

Explore two key fairness definitions:
- **Demographic parity**: P(ŷ=1|Group A) = P(ŷ=1|Group B) (equal positive prediction rate)
- **Equalized odds**: TPR and FPR equal across groups (equal error rates)

In [ ]:
# Compute predictions as probabilities
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Demographic parity: P(ŷ=1|group)
pred_rate_A = (y_pred[mask_A] == 1).mean()
pred_rate_B = (y_pred[mask_B] == 1).mean()

# Equalized odds: TPR and FPR per group
# Group A
cm_A = confusion_matrix(y_test[mask_A], y_pred[mask_A])
tpr_A = cm_A[1, 1] / (cm_A[1, 1] + cm_A[1, 0]) if (cm_A[1, 1] + cm_A[1, 0]) > 0 else 0
fpr_A = cm_A[0, 1] / (cm_A[0, 1] + cm_A[0, 0]) if (cm_A[0, 1] + cm_A[0, 0]) > 0 else 0

# Group B
cm_B = confusion_matrix(y_test[mask_B], y_pred[mask_B])
tpr_B = cm_B[1, 1] / (cm_B[1, 1] + cm_B[1, 0]) if (cm_B[1, 1] + cm_B[1, 0]) > 0 else 0
fpr_B = cm_B[0, 1] / (cm_B[0, 1] + cm_B[0, 0]) if (cm_B[0, 1] + cm_B[0, 0]) > 0 else 0

print("Fairness Metrics Analysis")
print("=" * 60)

print(f"\nDemographic Parity: P(ŷ=1|group)")
print(f"  Group A: {pred_rate_A:.4f}")
print(f"  Group B: {pred_rate_B:.4f}")
print(f"  Difference: {abs(pred_rate_A - pred_rate_B):.4f}")
if abs(pred_rate_A - pred_rate_B) > 0.1:
    print(f"  Status: VIOLATED (difference > 0.1)")
else:
    print(f"  Status: SATISFIED (difference <= 0.1)")

print(f"\nEqualized Odds: TPR and FPR per group")
print(f"  TPR (True Positive Rate, sensitivity):")
print(f"    Group A: {tpr_A:.4f}")
print(f"    Group B: {tpr_B:.4f}")
print(f"    Difference: {abs(tpr_A - tpr_B):.4f}")

print(f"\n  FPR (False Positive Rate, 1 - specificity):")
print(f"    Group A: {fpr_A:.4f}")
print(f"    Group B: {fpr_B:.4f}")
print(f"    Difference: {abs(fpr_A - fpr_B):.4f}")

if abs(tpr_A - tpr_B) > 0.1 or abs(fpr_A - fpr_B) > 0.1:
    print(f"\n  Status: VIOLATED (difference > 0.1)")
else:
    print(f"\n  Status: SATISFIED (differences <= 0.1)")

# Check if demographic parity and equalized odds conflict
demo_parity_gap = abs(pred_rate_A - pred_rate_B)
equalized_odds_gap = max(abs(tpr_A - tpr_B), abs(fpr_A - fpr_B))

print(f"\nConflict Analysis:")
print(f"  Demographic parity gap: {demo_parity_gap:.4f}")
print(f"  Equalized odds gap: {equalized_odds_gap:.4f}")
if demo_parity_gap < 0.05 and equalized_odds_gap > 0.1:
    print(f"  -> Demographic parity satisfied but equalized odds violated")
elif demo_parity_gap > 0.1 and equalized_odds_gap < 0.05:
    print(f"  -> Equalized odds satisfied but demographic parity violated")
elif demo_parity_gap < 0.05 and equalized_odds_gap < 0.05:
    print(f"  -> Both criteria satisfied (rare!)")
else:
    print(f"  -> Both criteria violated")

In [ ]:
# Visualize fairness metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Demographic parity
ax = axes[0]
groups_dp = ['Group A', 'Group B']
pred_rates = [pred_rate_A, pred_rate_B]
bars = ax.bar(groups_dp, pred_rates, color=['#2ca02c', '#d62728'], alpha=0.7, edgecolor='black')
ax.axhline((pred_rate_A + pred_rate_B) / 2, color='blue', linestyle='--', linewidth=2, label='Mean')
ax.set_ylabel('P(ŷ=1)', fontsize=11)
ax.set_title('Demographic Parity\nP(positive prediction | group)', fontweight='bold')
ax.set_ylim([0, 1.0])
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar, rate in zip(bars, pred_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.03, f'{rate:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Equalized odds - TPR
ax = axes[1]
tprs = [tpr_A, tpr_B]
bars = ax.bar(groups_dp, tprs, color=['#2ca02c', '#d62728'], alpha=0.7, edgecolor='black')
ax.axhline((tpr_A + tpr_B) / 2, color='blue', linestyle='--', linewidth=2, label='Mean')
ax.set_ylabel('TPR (Sensitivity)', fontsize=11)
ax.set_title('Equalized Odds: TPR\nTrue positive rate per group', fontweight='bold')
ax.set_ylim([0, 1.0])
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar, tpr in zip(bars, tprs):
    ax.text(bar.get_x() + bar.get_width() / 2, tpr + 0.03, f'{tpr:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Equalized odds - FPR
ax = axes[2]
fprs = [fpr_A, fpr_B]
bars = ax.bar(groups_dp, fprs, color=['#2ca02c', '#d62728'], alpha=0.7, edgecolor='black')
ax.axhline((fpr_A + fpr_B) / 2, color='blue', linestyle='--', linewidth=2, label='Mean')
ax.set_ylabel('FPR (1 - Specificity)', fontsize=11)
ax.set_title('Equalized Odds: FPR\nFalse positive rate per group', fontweight='bold')
ax.set_ylim([0, 1.0])
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar, fpr in zip(bars, fprs):
    ax.text(bar.get_x() + bar.get_width() / 2, fpr + 0.03, f'{fpr:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Demographic Parity vs. Equalized Odds', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Robustness Under Distribution Shift

Train on "clean" data (low noise) and test on "shifted" data (higher noise, different mean). Show how performance degrades.

In [ ]:
np.random.seed(42)

# Training data: clean
X_train_clean, y_train_clean = make_classification(
    n_samples=400,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    noise=1.0,
    random_state=42
)

# Test data 1: same distribution (clean)
X_test_clean, y_test_clean = make_classification(
    n_samples=200,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    noise=1.0,
    random_state=123
)

# Test data 2: shifted distribution (higher noise, mean shift)
X_test_shifted, y_test_shifted = make_classification(
    n_samples=200,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    noise=3.0,  # Higher noise
    random_state=124
)
X_test_shifted = X_test_shifted + 0.5  # Shift the mean

# Train model on clean training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train_clean)

# Evaluate on both test sets
X_test_clean_scaled = scaler.transform(X_test_clean)
X_test_shifted_scaled = scaler.transform(X_test_shifted)

acc_clean = model.score(X_test_clean_scaled, y_test_clean)
acc_shifted = model.score(X_test_shifted_scaled, y_test_shifted)

print(f"Robustness Under Distribution Shift")
print("=" * 60)
print(f"\nTraining Data: Clean (noise=1.0, n=400)")
print(f"Test Data 1 (same distribution):")
print(f"  Accuracy: {acc_clean:.4f}")
print(f"\nTest Data 2 (shifted distribution):")
print(f"  Higher noise (3.0), mean shift (+0.5)")
print(f"  Accuracy: {acc_shifted:.4f}")
print(f"\nAccuracy Drop: {acc_clean - acc_shifted:.4f} ({(acc_clean - acc_shifted)/acc_clean * 100:.1f}%)")
print(f"\nInsight: Model is sensitive to distribution shift!")

In [ ]:
# Visualize distribution shift
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Feature distributions
feature_idx = 0  # First feature

# Training vs clean test
ax = axes[0, 0]
ax.hist(X_train_clean[:, feature_idx], bins=20, alpha=0.6, label='Train (clean)', color='steelblue', edgecolor='black')
ax.hist(X_test_clean[:, feature_idx], bins=20, alpha=0.6, label='Test (clean)', color='orange', edgecolor='black')
ax.set_xlabel(f'Feature {feature_idx}')
ax.set_ylabel('Frequency')
ax.set_title('Training vs. Clean Test Distribution (ID)')
ax.legend()
ax.grid(alpha=0.3)

# Training vs shifted test
ax = axes[0, 1]
ax.hist(X_train_clean[:, feature_idx], bins=20, alpha=0.6, label='Train (clean)', color='steelblue', edgecolor='black')
ax.hist(X_test_shifted[:, feature_idx], bins=20, alpha=0.6, label='Test (shifted)', color='red', edgecolor='black')
ax.set_xlabel(f'Feature {feature_idx}')
ax.set_ylabel('Frequency')
ax.set_title('Training vs. Shifted Test Distribution (OOD)')
ax.legend()
ax.grid(alpha=0.3)

# Box plot comparison
ax = axes[1, 0]
data_to_plot = [X_train_clean[:, feature_idx], X_test_clean[:, feature_idx], X_test_shifted[:, feature_idx]]
bp = ax.boxplot(data_to_plot, labels=['Train (clean)', 'Test (clean)', 'Test (shifted)'],
                patch_artist=True)
colors = ['steelblue', 'orange', 'red']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel(f'Feature {feature_idx} Value')
ax.set_title('Distribution Comparison (Box Plot)')
ax.grid(axis='y', alpha=0.3)

# Accuracy comparison
ax = axes[1, 1]
test_types = ['Clean\n(ID)', 'Shifted\n(OOD)']
accuracies = [acc_clean, acc_shifted]
colors_acc = ['#2ca02c', '#d62728']
bars = ax.bar(test_types, accuracies, color=colors_acc, alpha=0.7, edgecolor='black')
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Model Accuracy: In-Distribution vs. Out-of-Distribution', fontweight='bold')
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.02, f'{acc:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Robustness Under Distribution Shift', fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 4. Spurious Correlations / Shortcuts

Train a model on data where a spurious feature (background color) correlates with the label in training but not in test. Show high train accuracy, low test accuracy on non-spurious data.

In [ ]:
np.random.seed(42)

# Training data: 10 real features + 1 spurious feature
n_train = 400
n_test = 200

# Real features
X_train_real, y_train = make_classification(
    n_samples=n_train,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

# Add spurious feature: perfectly correlated with label in training
spurious_train = y_train.copy().astype(float)
X_train_with_spurious = np.hstack([X_train_real, spurious_train.reshape(-1, 1)])

# Test data: real features from different distribution
X_test_real, y_test = make_classification(
    n_samples=n_test,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=99
)

# Spurious feature in test: random (no correlation with label)
spurious_test = np.random.randn(n_test)
X_test_with_spurious = np.hstack([X_test_real, spurious_test.reshape(-1, 1)])

# Model 1: Train with spurious feature
scaler1 = StandardScaler()
X_train_scaled = scaler1.fit_transform(X_train_with_spurious)
X_test_scaled = scaler1.transform(X_test_with_spurious)

model_with_spurious = LogisticRegression(random_state=42, max_iter=1000)
model_with_spurious.fit(X_train_scaled, y_train)

train_acc_with_spurious = model_with_spurious.score(X_train_scaled, y_train)
test_acc_with_spurious = model_with_spurious.score(X_test_scaled, y_test)

# Model 2: Train without spurious feature (only real features)
scaler2 = StandardScaler()
X_train_real_scaled = scaler2.fit_transform(X_train_real)
X_test_real_scaled = scaler2.transform(X_test_real)

model_without_spurious = LogisticRegression(random_state=42, max_iter=1000)
model_without_spurious.fit(X_train_real_scaled, y_train)

train_acc_without_spurious = model_without_spurious.score(X_train_real_scaled, y_train)
test_acc_without_spurious = model_without_spurious.score(X_test_real_scaled, y_test)

print("Spurious Correlations / Shortcuts")
print("=" * 60)
print(f"\nModel WITH spurious feature (background color):")
print(f"  Train accuracy: {train_acc_with_spurious:.4f}")
print(f"  Test accuracy:  {test_acc_with_spurious:.4f}")
print(f"  Overfitting gap: {train_acc_with_spurious - test_acc_with_spurious:.4f}")

print(f"\nModel WITHOUT spurious feature (only real features):")
print(f"  Train accuracy: {train_acc_without_spurious:.4f}")
print(f"  Test accuracy:  {test_acc_without_spurious:.4f}")
print(f"  Overfitting gap: {train_acc_without_spurious - test_acc_without_spurious:.4f}")

print(f"\nTest accuracy improvement by removing spurious feature:")
print(f"  {test_acc_without_spurious - test_acc_with_spurious:+.4f}")

# Feature importance from the model with spurious feature
coef_spurious = model_with_spurious.coef_[0]
print(f"\nModel WITH spurious feature - coefficient magnitudes:")
print(f"  Real features (first 10): {np.mean(np.abs(coef_spurious[:10])):.4f} (mean)")
print(f"  Spurious feature: {np.abs(coef_spurious[10]):.4f}")
print(f"  -> Spurious feature has HIGH coefficient (but misleading!)")

In [ ]:
# Visualize spurious correlation impact
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Train vs Test accuracy
ax = axes[0]
x = np.arange(2)
width = 0.35

train_accs = [train_acc_with_spurious, train_acc_without_spurious]
test_accs = [test_acc_with_spurious, test_acc_without_spurious]

bars1 = ax.bar(x - width/2, train_accs, width, label='Train', color='steelblue', alpha=0.7)
bars2 = ax.bar(x + width/2, test_accs, width, label='Test', color='#d62728', alpha=0.7)

ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Impact of Spurious Correlations', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['With Spurious\nFeature', 'Without Spurious\nFeature'])
ax.set_ylim([0, 1.0])
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height + 0.02,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# Overfitting gap
ax = axes[1]
gaps = [train_acc_with_spurious - test_acc_with_spurious, 
        train_acc_without_spurious - test_acc_without_spurious]
colors_gap = ['#d62728', '#2ca02c']
bars = ax.bar(['With Spurious', 'Without Spurious'], gaps, color=colors_gap, alpha=0.7, edgecolor='black')
ax.set_ylabel('Train - Test Accuracy', fontsize=11)
ax.set_title('Overfitting Gap: Generalization Failure', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width() / 2, gap + 0.01, f'{gap:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Spurious Features Lead to Poor Generalization', fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 5. Calibration Under Shift

Train a well-calibrated model and show that calibration degrades when the data distribution shifts.

In [ ]:
np.random.seed(42)

# Generate training data and calibration set
X_train, y_train = make_classification(
    n_samples=400,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

# Calibration data: same distribution
X_cal, y_cal = make_classification(
    n_samples=200,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=45
)

# Test data 1: same distribution (in-distribution)
X_test_id, y_test_id = make_classification(
    n_samples=200,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=46
)

# Test data 2: shifted distribution (out-of-distribution)
X_test_ood, y_test_ood = make_classification(
    n_samples=200,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=47
)
X_test_ood = X_test_ood + 0.8  # Shift mean

# Train model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Get predictions on calibration, ID test, and OOD test
X_cal_scaled = scaler.transform(X_cal)
X_test_id_scaled = scaler.transform(X_test_id)
X_test_ood_scaled = scaler.transform(X_test_ood)

y_pred_cal = model.predict_proba(X_cal_scaled)[:, 1]
y_pred_id = model.predict_proba(X_test_id_scaled)[:, 1]
y_pred_ood = model.predict_proba(X_test_ood_scaled)[:, 1]

# Compute calibration curves
def calibration_curve(y_true, y_pred_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_accs = []
    bin_confs = []
    bin_counts = []
    
    for i in range(n_bins):
        mask = (y_pred_proba >= bins[i]) & (y_pred_proba < bins[i+1])
        if mask.sum() > 0:
            bin_acc = y_true[mask].mean()
            bin_conf = y_pred_proba[mask].mean()
            bin_accs.append(bin_acc)
            bin_confs.append(bin_conf)
            bin_counts.append(mask.sum())
        else:
            bin_accs.append(np.nan)
            bin_confs.append(bin_centers[i])
            bin_counts.append(0)
    
    return np.array(bin_confs), np.array(bin_accs), np.array(bin_counts)

conf_id, acc_id, counts_id = calibration_curve(y_test_id, y_pred_id)
conf_ood, acc_ood, counts_ood = calibration_curve(y_test_ood, y_pred_ood)

# Compute expected calibration error (ECE)
def ece(y_true, y_pred_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece_val = 0
    for i in range(n_bins):
        mask = (y_pred_proba >= bins[i]) & (y_pred_proba < bins[i+1])
        if mask.sum() > 0:
            acc = y_true[mask].mean()
            conf = y_pred_proba[mask].mean()
            ece_val += np.abs(acc - conf) * mask.sum() / len(y_true)
    return ece_val

ece_id = ece(y_test_id, y_pred_id)
ece_ood = ece(y_test_ood, y_pred_ood)

print("Calibration Under Distribution Shift")
print("=" * 60)
print(f"\nIn-Distribution Test (ID):")
print(f"  Expected Calibration Error (ECE): {ece_id:.4f}")
print(f"  Interpretation: Model is {'well-calibrated' if ece_id < 0.05 else 'poorly calibrated'} (ECE {'<' if ece_id < 0.05 else '>'} 0.05)")

print(f"\nOut-of-Distribution Test (OOD):")
print(f"  Expected Calibration Error (ECE): {ece_ood:.4f}")
print(f"  Interpretation: Model is {'well-calibrated' if ece_ood < 0.05 else 'poorly calibrated'} (ECE {'<' if ece_ood < 0.05 else '>'} 0.05)")

print(f"\nCalibration Degradation:")
print(f"  ECE increase: {ece_ood - ece_id:+.4f}")
print(f"  -> Calibration degrades under distribution shift")

In [ ]:
# Plot reliability diagrams
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# In-distribution
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect calibration')
ax.plot(conf_id, acc_id, 'o-', color='steelblue', linewidth=2, markersize=7, label='Model')
ax.fill_between(conf_id, conf_id, acc_id, alpha=0.3, color='steelblue')
ax.set_xlabel('Predicted Probability (Confidence)', fontsize=11)
ax.set_ylabel('Empirical Probability (Accuracy)', fontsize=11)
ax.set_title(f'In-Distribution Calibration\nECE = {ece_id:.4f}', fontweight='bold')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Out-of-distribution
ax = axes[1]
ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect calibration')
ax.plot(conf_ood, acc_ood, 'o-', color='#d62728', linewidth=2, markersize=7, label='Model')
ax.fill_between(conf_ood, conf_ood, acc_ood, alpha=0.3, color='#d62728')
ax.set_xlabel('Predicted Probability (Confidence)', fontsize=11)
ax.set_ylabel('Empirical Probability (Accuracy)', fontsize=11)
ax.set_title(f'Out-of-Distribution Calibration\nECE = {ece_ood:.4f}', fontweight='bold')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.suptitle('Reliability Diagrams: Calibration Degradation Under Shift', 
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## What to Notice

1. **Subgroup Evaluation Is Critical**: Overall metrics hide subgroup failures. Always compute performance separately for meaningful population segments.

2. **Fairness Criteria Can Conflict**: Demographic parity and equalized odds are often incompatible. Choose the right fairness definition for your context.

3. **Distribution Shift Breaks Models**: A model trained on clean data performs poorly on shifted distributions. This is a fundamental challenge in deployment.

4. **Spurious Correlations Are Invisible**: The model happily learns to exploit spurious features in training. These cause catastrophic failure on test data without the correlation.

5. **Calibration Degrades Under Shift**: Even if accuracy holds, calibration (confidence in predictions) becomes unreliable. Trust predicted probabilities only for in-distribution data.

6. **Reliability Requires Vigilance**: Building reliable ML systems demands active monitoring of subgroup performance, distribution shifts, and calibration metrics.